### INSTALAÇÕES - SE NECESSÁRIO -




In [51]:
# Instalar pacotes necessários (se ainda não instalados)
!pip install nltk scikit-learn

In [52]:
!pip install Unidecode

### IMPORTAÇÕES

In [53]:
import glob
import json
import pandas as pd
import re
import unidecode
import nltk
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import os

In [54]:
# prompt: conecte ao drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### TRATAMENTO DOS DADOS

In [55]:
# Configurar NLTK (stopwords e stemmer)

# Baixar recursos do NLTK
nltk.download('stopwords')
nltk.download('rslp')

# Configurações
stopwords_pt = set(stopwords.words('portuguese'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Package rslp is already up-to-date!


Opção 2 de pré processamento

In [56]:
# Lista de stopwords padrão do português
stopwords_pt = set(stopwords.words('portuguese'))

# Stemmer
stemmer = RSLPStemmer()

# Lista de palavras-chave (cidades e candidatos) a serem excluídas
palavras_excluir = {
    'aracaju', 'emilia', 'correa', 'luiz', 'roberto',
    'fortaleza', 'andre', 'fernandes', 'evandro', 'leitao',
    'joao', 'pessoa', 'cicero', 'lucena', 'luciano', 'cartaxo',
    'maceio', 'jhc', 'rafael', 'brito',
    'natal', 'natalia', 'bonavides', 'paulinho', 'freire',
    'recife', 'gilson', 'machado', 'neto', 'campos',
    'salvador', 'bruno', 'reis', 'geraldo', 'jr', 'kleber', 'rosa',
    'sao', 'luis', 'duarte', 'eduardo', 'braide',
    'teresina', 'fabio', 'novo', 'silvio', 'mendes',  'marcelo', 'queiroga', 'propaganda', 'eleitoral'
}

def preprocessar_texto(texto):
    if pd.isna(texto) or texto.strip() == '':
        return ''

    texto = texto.lower()

    # Remover URLs
    texto = re.sub(r"http\S+|www\S+", '', texto)

    # Remover hashtags e menções
    texto = re.sub(r'#\S+', '', texto)
    texto = re.sub(r'@\S+', '', texto)

    # Remover caracteres não alfabéticos
    texto = re.sub(r'[^a-zA-ZÀ-ÿ\s]', '', texto)

    # Remover acentuação
    texto = unidecode.unidecode(texto)

    # Tokenização
    palavras = texto.split()

    # Remover stopwords e palavras a excluir
    palavras_filtradas = [p for p in palavras if p not in stopwords_pt and p not in palavras_excluir]

    return ' '.join(palavras_filtradas)


## Direita

### TRATAMENTO

In [57]:
# Carregar todos os JSONs da pasta
caminho_json_direita = "/content/drive/My Drive/Colab Notebooks/ENIAC/2.0/eixo/direita/*.json"
arquivos_direita = glob.glob(caminho_json_direita)

# Lista para armazenar os DataFrames de cada arquivo
dfs_direita = []

for arquivo in arquivos_direita:
    with open(arquivo, 'r', encoding='utf-8') as f:
        posts = json.load(f)

        df = pd.DataFrame(posts)

        # Criar coluna com texto pré-processado
        df['texto_processado'] = df['caption'].apply(preprocessar_texto)

        # Manter SOMENTE a coluna processada
        df = df[['texto_processado']]

        dfs_direita.append(df)

In [58]:
# Criar diretório para salvar os CSVs, se não existir
os.makedirs("dataset_ideologia", exist_ok=True)

# Concatenar todos os DataFrames em um único DataFrame para Direita
df_direita_final = pd.concat(dfs_direita, ignore_index=True)

# Salvar o DataFrame final como CSV
nome_arquivo_csv = "dataset_ideologia/direita_processado.csv"
df_direita_final.to_csv(nome_arquivo_csv, index=False, encoding='utf-8')

print(f"Dataset da ideologia Direita exportado para: {nome_arquivo_csv}")

Dataset da ideologia Direita exportado para: dataset_ideologia/direita_processado.csv


In [59]:
# Aplicar TF-IDF
vectorizer = TfidfVectorizer(max_features=1000)
X_tfidf = vectorizer.fit_transform(df_direita_final['texto_processado'])

# Obter termos filtrados
termos_filtrados = vectorizer.get_feature_names_out()

In [60]:
# Converter a matriz esparsa TF-IDF para um DataFrame denso
df_tfidf_direita = pd.DataFrame(X_tfidf.toarray(), columns=termos_filtrados)

# Salvar o DataFrame TF-IDF como CSV
nome_arquivo_tfidf_csv = "dataset_ideologia/dataset_TF-IDF_direita.csv"
df_tfidf_direita.to_csv(nome_arquivo_tfidf_csv, index=False, encoding='utf-8')

print(f"Dataset TF-IDF da ideologia Direita exportado para: {nome_arquivo_tfidf_csv}")

Dataset TF-IDF da ideologia Direita exportado para: dataset_ideologia/dataset_TF-IDF_direita.csv


### WORDCLOUDS

In [ ]:
os.makedirs("wordclouds", exist_ok=True)

# Somar os pesos TF-IDF por termo (across all documents)
somas_tfidf = X_tfidf.sum(axis=0).A1
pesos_por_termo = dict(zip(termos_filtrados, somas_tfidf))

# Criar WordCloud
wordcloud = WordCloud(width=1000, height=500, background_color='white', colormap='Dark2').generate_from_frequencies(pesos_por_termo)

# Mostrar
plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("WordCloud - Direita", fontsize=18)
# Substituir espaços ou caracteres inválidos no nome do arquivo
nome_arquivo = f"wordclouds/direita.png"
plt.savefig(nome_arquivo, bbox_inches='tight')
plt.close()  # Fecha a figura para economizar memória


In [ ]:
# Imprimir os 10 termos mais frequentes
# Abre (ou cria) o arquivo txt para escrita
with open("top_10_termos_direita.txt", "w", encoding="utf-8") as f:
  # Já temos: pesos_por_termo = dict(zip(termos_filtrados, somas_tfidf))

  # Ordenar os termos por peso TF-IDF (do maior para o menor)
  termos_ordenados = sorted(pesos_por_termo.items(), key=lambda x: x[1], reverse=True)

  # Imprimir os 10 primeiros
  f.write("Top 10 termos mais frequentes:\n")
  for termo, peso in termos_ordenados[:10]:
      f.write(f"{termo} ({peso:.3f})\n")


## Esquerda

### TRATAMENTO

In [ ]:
# Carregar todos os JSONs da pasta
caminho_json_esquerda = "/content/drive/My Drive/Colab Notebooks/ENIAC/2.0/eixo/esquerda/*.json"
arquivos_esquerda = glob.glob(caminho_json_esquerda)

# Lista para armazenar os DataFrames de cada arquivo
dfs_esquerda = []

for arquivo in arquivos_esquerda:
    with open(arquivo, 'r', encoding='utf-8') as f:
        posts = json.load(f)

        df = pd.DataFrame(posts)

        # Criar coluna com texto pré-processado
        df['texto_processado'] = df['caption'].apply(preprocessar_texto)

        # Manter SOMENTE a coluna processada
        df = df[['texto_processado']]

        dfs_esquerda.append(df)


In [ ]:
# Criar diretório para salvar os CSVs, se não existir
os.makedirs("dataset_ideologia", exist_ok=True)

# Concatenar todos os DataFrames em um único DataFrame para Direita
df_esquerda_final = pd.concat(dfs_esquerda, ignore_index=True)

# Salvar o DataFrame final como CSV
nome_arquivo_csv = "dataset_ideologia/esquerda_processado.csv"
df_esquerda_final.to_csv(nome_arquivo_csv, index=False, encoding='utf-8')

print(f"Dataset da ideologia Esquerda exportado para: {nome_arquivo_csv}")

Dataset da ideologia Esquerda exportado para: dataset_ideologia/esquerda_processado.csv


In [ ]:
# Aplicar TF-IDF
vectorizer = TfidfVectorizer(max_features=1000)
X_tfidf = vectorizer.fit_transform(df_esquerda_final['texto_processado'])

# Obter termos filtrados
termos_filtrados = vectorizer.get_feature_names_out()

In [ ]:
# Converter a matriz esparsa TF-IDF para um DataFrame denso
df_tfidf_esquerda = pd.DataFrame(X_tfidf.toarray(), columns=termos_filtrados)

# Salvar o DataFrame TF-IDF como CSV
nome_arquivo_tfidf_csv = "dataset_ideologia/dataset_TF-IDF_esquerda.csv"
df_tfidf_esquerda.to_csv(nome_arquivo_tfidf_csv, index=False, encoding='utf-8')

print(f"Dataset TF-IDF da ideologia Esquerda exportado para: {nome_arquivo_tfidf_csv}")

Dataset TF-IDF da ideologia Esquerda exportado para: dataset_ideologia/dataset_TF-IDF_esquerda.csv


### WORDCLOUDS

In [ ]:
os.makedirs("wordclouds", exist_ok=True)

# Somar os pesos TF-IDF por termo (across all documents)
somas_tfidf = X_tfidf.sum(axis=0).A1
pesos_por_termo = dict(zip(termos_filtrados, somas_tfidf))

# Criar WordCloud
wordcloud = WordCloud(width=1000, height=500, background_color='white', colormap='Dark2').generate_from_frequencies(pesos_por_termo)

# Mostrar
plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("WordCloud - Esquerda", fontsize=18)
# Substituir espaços ou caracteres inválidos no nome do arquivo
nome_arquivo = f"wordclouds/esquerda.png"
plt.savefig(nome_arquivo, bbox_inches='tight')
plt.close()  # Fecha a figura para economizar memória


In [ ]:
# Imprimir os 10 termos mais frequentes
# Abre (ou cria) o arquivo txt para escrita
with open("top_10_termos_esquerda.txt", "w", encoding="utf-8") as f:
  # Já temos: pesos_por_termo = dict(zip(termos_filtrados, somas_tfidf))

  # Ordenar os termos por peso TF-IDF (do maior para o menor)
  termos_ordenados = sorted(pesos_por_termo.items(), key=lambda x: x[1], reverse=True)

  # Imprimir os 10 primeiros
  f.write("Top 10 termos mais frequentes:\n")
  for termo, peso in termos_ordenados[:10]:
      f.write(f"{termo} ({peso:.3f})\n")
